# Exploring the American put pricers

Quick, informal look at the three pricing methods on one contract before jumping into the full grid comparison in `evaluation/comparison.py`. Run this from the repo root with the package installed (`pip install -e .`).

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import matplotlib.pyplot as plt

from pricing.binomial import crr_price, exercise_boundary_by_step
from pricing.black_scholes import black_scholes_price
from ml.evaluate_nn import NNPricer
from data.synthetic_contracts import OptionContract

## One contract, three ways

In [2]:
contract = OptionContract(S0=100, K=100, T=1.0, r=0.05, sigma=0.25, steps=500)

american = crr_price(contract.S0, contract.K, contract.T, contract.r, contract.sigma, contract.steps, american=True)
european = crr_price(contract.S0, contract.K, contract.T, contract.r, contract.sigma, contract.steps, american=False)
bs = black_scholes_price(contract.S0, contract.K, contract.T, contract.r, contract.sigma)

print(f'binomial American : {american:.4f}')
print(f'binomial European  : {european:.4f}')
print(f'Black-Scholes (Eur): {bs:.4f}')
print(f'early-exercise premium: {american - european:.4f}')

binomial American : 7.9724
binomial European  : 7.4540
Black-Scholes (Eur): 7.4589
early-exercise premium: 0.5184


In [3]:
nn = NNPricer()  # loads reports/results/neural_pricer.pt -- run `python -m ml.train_nn` first if missing
print(f'NN predicted price : {nn.predict(contract):.4f}')

NN predicted price : 8.0556


## Where does the tree say to exercise?

The boundary is the highest stock price, at each point in time, where exercising beats holding. It should climb toward the strike as expiry approaches.

In [4]:
boundary = exercise_boundary_by_step(contract.S0, contract.K, contract.T, contract.r, contract.sigma, steps=200)
dt = contract.T / 200
steps_sorted = sorted(boundary)

plt.figure(figsize=(6,4))
plt.plot([i*dt for i in steps_sorted], [boundary[i] for i in steps_sorted], marker='.')
plt.axhline(contract.K, color='gray', ls='--', label='strike')
plt.xlabel('time'); plt.ylabel('exercise boundary (spot)'); plt.legend()
plt.title('When does early exercise kick in?');

## Does the NN respect the shape of a put?

Sweeping spot at fixed K/T/r/sigma -- price should fall monotonically as spot rises, and should never dip below max(K-S, 0).

In [5]:
spots = np.linspace(60, 140, 41)
nn_prices = [nn.predict(OptionContract(S0=s, K=100, T=1.0, r=0.05, sigma=0.25, steps=100)) for s in spots]
binomial_prices = [crr_price(s, 100, 1.0, 0.05, 0.25, 200, american=True) for s in spots]

plt.figure(figsize=(6,4))
plt.plot(spots, binomial_prices, label='binomial')
plt.plot(spots, nn_prices, label='NN', ls='--')
plt.plot(spots, np.maximum(100-spots, 0), label='intrinsic', color='gray', ls=':')
plt.xlabel('spot'); plt.ylabel('price'); plt.legend()
plt.title('NN vs binomial as spot varies');